# Electricity Foundation Models

Phase 5 evaluates `amazon/chronos-bolt-tiny` and `google/timesfm-2.5-200m-pytorch` as zero-shot models on South Australian half-hourly demand. The frozen context is 336 observations; Protocol A is rolling one-step and Protocol B is a true 48-step day-ahead forecast. No model is fine-tuned.

## 1. Load Frozen Electricity Data

In [1]:
from pathlib import Path
import sys, time, platform, importlib.metadata as md, gc
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import torch, psutil

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing src/")

ROOT=find_project_root(Path.cwd()); DATA=ROOT/"data/electricity/australian_electricity_demand_dataset.tsf"; RESULTS=ROOT/"results/electricity"
RESULTS.mkdir(parents=True,exist_ok=True)
def load_tsf(path):
    attrs=[]; rows=[]
    with Path(path).open(encoding="utf-8") as f:
        for raw in f:
            line=raw.strip()
            if not line or line.startswith("#"): continue
            if line.startswith("@attribute"):
                _,n,k=line.split(maxsplit=2); attrs.append((n,k))
            elif not line.startswith("@"):
                p=line.split(":",len(attrs)); r=dict(zip((a[0] for a in attrs),p[:-1])); r["series_value"]=np.fromstring(p[-1],sep=","); rows.append(r)
    return pd.DataFrame(rows)
raw=load_tsf(DATA); selected=raw[(raw.series_name=="T4")&(raw.state=="SA")]; assert len(selected)==1
row=selected.iloc[0]; idx=pd.date_range(pd.to_datetime(row.start_timestamp,format="%Y-%m-%d %H-%M-%S"),periods=len(row.series_value),freq="30min")
y=pd.Series(row.series_value,index=idx,name="Actual"); dev=y.loc[:"2011-06-23 23:30"]; validation=y.loc["2011-06-24":"2012-07-12 23:30"]
pretest=y.loc[:"2012-07-12 23:30"]; test=y.loc["2012-07-13":"2015-03-01 23:30"]
assert (len(dev),len(validation),len(pretest),len(test))==(166128,18480,184608,46176)
assert test.index.equals(pd.date_range("2012-07-13",periods=46176,freq="30min"))
scale48=float(np.mean(np.abs(pretest.to_numpy()[48:]-pretest.to_numpy()[:-48])))
display(pd.DataFrame({"Partition":["Development","Validation","Pre-test","Final test"],"Start":[z.index[0] for z in [dev,validation,pretest,test]],"End":[z.index[-1] for z in [dev,validation,pretest,test]],"N":[len(z) for z in [dev,validation,pretest,test]]}))
print("Selected:",row.series_name,row.state,"MASE-48 denominator:",scale48)

## 2. Frozen Forecasting Protocols

In [2]:
CONTEXT=336; HORIZON=48; N_A=len(test); origins=test.index[::48]; assert len(origins)==962
values=y.to_numpy(float); positions=pd.Series(np.arange(len(y)),index=y.index)
test_positions=positions.loc[test.index].to_numpy(); contexts_a=np.stack([values[p-CONTEXT:p] for p in test_positions]).astype(np.float32)
origin_positions=positions.loc[origins].to_numpy(); contexts_b=np.stack([values[p-CONTEXT:p] for p in origin_positions]).astype(np.float32)
assert contexts_a.shape==(46176,336) and contexts_b.shape==(962,336)
print("Protocol A:",contexts_a.shape,"one forecast per target, actual revealed only afterward")
print("Protocol B:",contexts_b.shape,"one 48-step operation per midnight origin, no within-day updates")

## 3. Environment and Model Setup

In [3]:
env=pd.Series({"Python":sys.version,"Interpreter":sys.executable,"PyTorch":torch.__version__,"CUDA available":torch.cuda.is_available(),"Total RAM GiB":psutil.virtual_memory().total/2**30,"Available RAM GiB":psutil.virtual_memory().available/2**30,"chronos-forecasting":md.version("chronos-forecasting"),"timesfm":md.version("timesfm"),"Chronos model":"amazon/chronos-bolt-tiny","TimesFM model":"google/timesfm-2.5-200m-pytorch","Context":CONTEXT})
display(env.to_frame("Value"))
def batches(a,n):
    for i in range(0,len(a),n): yield i,a[i:i+n]
runtime={}; smoke={}; uncertainty={}
def metrics(actual,pred):
    a=np.asarray(actual,float); p=np.asarray(pred,float); e=a-p
    return {"MAE":np.mean(np.abs(e)),"RMSE":np.sqrt(np.mean(e**2)),"MAPE":np.mean(np.abs(e/a))*100,"sMAPE":np.mean(2*np.abs(e)/(np.abs(a)+np.abs(p)))*100,"MASE_48":np.mean(np.abs(e))/scale48}
def diagnostics(actual,pred):
    a=np.asarray(actual,float); p=np.asarray(pred,float)
    return {"Prediction_Min":p.min(),"Prediction_Max":p.max(),"Prediction_Std":p.std(ddof=1),"Actual_Std":a.std(ddof=1),"Prediction_Actual_Std_Ratio":p.std(ddof=1)/a.std(ddof=1),"Change_Std_Ratio":np.diff(p).std(ddof=1)/np.diff(a).std(ddof=1),"Correlation":np.corrcoef(a,p)[0,1],"Constant":np.unique(p).size<=1,"Range_Compression_Ratio":np.ptp(p)/np.ptp(a),"Finite":np.isfinite(p).all()}

## 4. Chronos Smoke Test

In [4]:
from chronos import ChronosBoltPipeline
ram0=psutil.virtual_memory().available/2**30; t=time.perf_counter(); chronos_model=ChronosBoltPipeline.from_pretrained("amazon/chronos-bolt-tiny",device_map="cpu",dtype=torch.float32); runtime["Chronos_load_s"]=time.perf_counter()-t; ram1=psutil.virtual_memory().available/2**30
chronos_quantiles=list(map(float,chronos_model.quantiles)); print("Verified Chronos quantiles:",chronos_quantiles)
t=time.perf_counter(); raw_ch=chronos_model.predict(torch.tensor(contexts_b[:1]),prediction_length=48); dt=time.perf_counter()-t
raw_ch_np=raw_ch.detach().cpu().numpy(); point_ch=raw_ch_np[:,chronos_quantiles.index(0.5),:]
smoke["Chronos"]={"raw_type":str(type(raw_ch)),"raw_shape":raw_ch_np.shape,"point_shape":point_ch.shape,"first_10":point_ch[0,:10],"min":point_ch.min(),"max":point_ch.max(),"mean":point_ch.mean(),"std":point_ch.std(),"finite":np.isfinite(point_ch).all(),"constant":np.unique(point_ch).size<=1,"load_s":runtime["Chronos_load_s"],"inference_s":dt,"RAM_before_GiB":ram0,"RAM_after_GiB":ram1}
display(pd.Series(smoke["Chronos"]).to_frame("Value")); assert smoke["Chronos"]["finite"] and not smoke["Chronos"]["constant"] and point_ch.shape==(1,48)
plt.figure(figsize=(12,4)); plt.plot(range(-96,0),contexts_b[0,-96:],label="Context"); plt.plot(range(48),point_ch[0],label="Forecast"); plt.legend(); plt.title("Chronos smoke test"); plt.show()

## 5. TimesFM Smoke Test

In [5]:
import timesfm
ram0=psutil.virtual_memory().available/2**30; t=time.perf_counter(); timesfm_model=timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch",torch_compile=False)
timesfm_model.compile(timesfm.ForecastConfig(max_context=336,max_horizon=48,normalize_inputs=True,per_core_batch_size=256)); runtime["TimesFM_load_s"]=time.perf_counter()-t; ram1=psutil.virtual_memory().available/2**30
print("TimesFM object quantile metadata:",{k:v for k,v in vars(timesfm_model).items() if "quant" in k.lower()})
t=time.perf_counter(); raw_tf=timesfm_model.forecast(horizon=48,inputs=[contexts_b[0]]); dt=time.perf_counter()-t
tf_point,tf_quant=map(np.asarray,raw_tf); print("TimesFM raw tuple types:",[type(x) for x in raw_tf],"shapes:",[x.shape for x in raw_tf])
tf_quantiles=list(map(float,getattr(timesfm_model,"quantiles",[.1,.2,.3,.4,.5,.6,.7,.8,.9])))
print("Verified TimesFM quantiles:",tf_quantiles)
smoke["TimesFM"]={"raw_type":str(type(raw_tf)),"raw_shape":(tf_point.shape,tf_quant.shape),"point_shape":tf_point.shape,"first_10":tf_point[0,:10],"min":tf_point.min(),"max":tf_point.max(),"mean":tf_point.mean(),"std":tf_point.std(),"finite":np.isfinite(tf_point).all(),"constant":np.unique(tf_point).size<=1,"load_s":runtime["TimesFM_load_s"],"inference_s":dt,"RAM_before_GiB":ram0,"RAM_after_GiB":ram1}
display(pd.Series(smoke["TimesFM"]).to_frame("Value")); assert smoke["TimesFM"]["finite"] and not smoke["TimesFM"]["constant"] and tf_point.shape==(1,48)
plt.figure(figsize=(12,4)); plt.plot(range(-96,0),contexts_b[0,-96:],label="Context"); plt.plot(range(48),tf_point[0],label="Forecast"); plt.legend(); plt.title("TimesFM smoke test"); plt.show()

## 6. Protocol A — Chronos

In [6]:
ch_ck=RESULTS/".phase5_chronos_a_checkpoint.npz"; ch_a=[]; ch_a_lo=[]; ch_a_hi=[]; start=0
if ch_ck.exists():
    z=np.load(ch_ck); ch_a=[z["point"]]; ch_a_lo=[z["lo"]]; ch_a_hi=[z["hi"]]; start=len(ch_a[0]); print("Resuming Chronos A at",start)
t=time.perf_counter()
for i,batch in batches(contexts_a[start:],1024):
    out=chronos_model.predict(torch.from_numpy(batch),prediction_length=1).detach().cpu().numpy()
    ch_a.append(out[:,chronos_quantiles.index(.5),0]); ch_a_lo.append(out[:,chronos_quantiles.index(.1),0]); ch_a_hi.append(out[:,chronos_quantiles.index(.9),0])
    done=start+i+len(batch)
    if done==len(contexts_a) or ((i//1024+1)%5==0): np.savez(ch_ck,point=np.concatenate(ch_a),lo=np.concatenate(ch_a_lo),hi=np.concatenate(ch_a_hi))
runtime["Chronos_A_s"]=time.perf_counter()-t; ch_a=np.concatenate(ch_a); ch_a_lo=np.concatenate(ch_a_lo); ch_a_hi=np.concatenate(ch_a_hi)
chronos_a_stats={**metrics(test,ch_a),**diagnostics(test,ch_a),"Inference_s":runtime["Chronos_A_s"],"Seconds_per_forecast":runtime["Chronos_A_s"]/len(test)}
pa_ch=pd.DataFrame({"Timestamp":test.index,"Chronos_Bolt_Tiny":ch_a}); pa_ch.to_csv(RESULTS/"protocol_a_chronos_forecast.csv",index=False,date_format="%Y-%m-%d %H:%M:%S"); display(pd.Series(chronos_a_stats).to_frame("Chronos"))

## 7. Protocol A — TimesFM

In [7]:
tf_ck=RESULTS/".phase5_timesfm_a_checkpoint.npz"; tf_a=[]; tf_a_lo=[]; tf_a_hi=[]; start=0
if tf_ck.exists():
    z=np.load(tf_ck); tf_a=[z["point"]]; tf_a_lo=[z["lo"]]; tf_a_hi=[z["hi"]]; start=len(tf_a[0]); print("Resuming TimesFM A at",start)
t=time.perf_counter()
for i,batch in batches(contexts_a[start:],256):
    point,q=timesfm_model.forecast(horizon=1,inputs=[x for x in batch]); point=np.asarray(point); q=np.asarray(q); tf_a.append(point[:,0]); tf_a_lo.append(q[:,0,tf_quantiles.index(.1)]); tf_a_hi.append(q[:,0,tf_quantiles.index(.9)])
    done=start+i+len(batch)
    if done==len(contexts_a) or ((i//256+1)%10==0): np.savez(tf_ck,point=np.concatenate(tf_a),lo=np.concatenate(tf_a_lo),hi=np.concatenate(tf_a_hi))
runtime["TimesFM_A_s"]=time.perf_counter()-t; tf_a=np.concatenate(tf_a); tf_a_lo=np.concatenate(tf_a_lo); tf_a_hi=np.concatenate(tf_a_hi)
timesfm_a_stats={**metrics(test,tf_a),**diagnostics(test,tf_a),"Inference_s":runtime["TimesFM_A_s"],"Seconds_per_forecast":runtime["TimesFM_A_s"]/len(test)}
pa_tf=pd.DataFrame({"Timestamp":test.index,"TimesFM":tf_a}); pa_tf.to_csv(RESULTS/"protocol_a_timesfm_forecast.csv",index=False,date_format="%Y-%m-%d %H:%M:%S"); display(pd.Series(timesfm_a_stats).to_frame("TimesFM"))

## 8. Protocol A Comparison

In [8]:
pa_ch=pd.DataFrame({"Timestamp":test.index,"Chronos_Bolt_Tiny":ch_a}); pa_tf=pd.DataFrame({"Timestamp":test.index,"TimesFM":tf_a})
pa=pd.read_csv(RESULTS/"protocol_a_baseline_forecasts.csv",parse_dates=["Timestamp"]); dhr=pd.read_csv(RESULTS/"protocol_a_dhr_forecast.csv",parse_dates=["Timestamp"]); lstm=pd.read_csv(RESULTS/"protocol_a_lstm_forecast.csv",parse_dates=["Timestamp"])
pa=pa.merge(dhr,on="Timestamp",validate="one_to_one").merge(lstm,on="Timestamp",validate="one_to_one").merge(pa_ch,on="Timestamp",validate="one_to_one").merge(pa_tf,on="Timestamp",validate="one_to_one")
name_a={"DHR_ARIMA":"DHR-ARIMA","Naive":"Naive","LSTM":"LSTM","Chronos_Bolt_Tiny":"Chronos-Bolt-Tiny","TimesFM":"TimesFM","Daily_Seasonal_Naive":"Daily Seasonal Naive","Weekly_Seasonal_Naive":"Weekly Seasonal Naive","Moving_Average":"Moving Average"}
rank_a=pd.DataFrame([{"Model":label,**metrics(pa.Actual,pa[col])} for col,label in name_a.items()]).sort_values("MASE_48").reset_index(drop=True); display(rank_a)

## 9. Protocol B — Chronos

In [9]:
ch_b=[]; ch_b_lo=[]; ch_b_hi=[]; t=time.perf_counter()
for _,batch in batches(contexts_b,256):
    out=chronos_model.predict(torch.from_numpy(batch),prediction_length=48).detach().cpu().numpy(); ch_b.append(out[:,chronos_quantiles.index(.5),:]); ch_b_lo.append(out[:,chronos_quantiles.index(.1),:]); ch_b_hi.append(out[:,chronos_quantiles.index(.9),:])
runtime["Chronos_B_s"]=time.perf_counter()-t; ch_b=np.concatenate(ch_b); ch_b_lo=np.concatenate(ch_b_lo); ch_b_hi=np.concatenate(ch_b_hi)
chronos_b_stats={**metrics(test,ch_b.ravel()),**diagnostics(test,ch_b.ravel()),"Inference_s":runtime["Chronos_B_s"],"Seconds_per_origin":runtime["Chronos_B_s"]/len(origins)}; display(pd.Series(chronos_b_stats).to_frame("Chronos"))

## 10. Protocol B — TimesFM

In [10]:
tf_b=[]; tf_b_lo=[]; tf_b_hi=[]; t=time.perf_counter()
for _,batch in batches(contexts_b,256):
    point,q=timesfm_model.forecast(horizon=48,inputs=[x for x in batch]); point=np.asarray(point); q=np.asarray(q); tf_b.append(point); tf_b_lo.append(q[:,:,tf_quantiles.index(.1)]); tf_b_hi.append(q[:,:,tf_quantiles.index(.9)])
runtime["TimesFM_B_s"]=time.perf_counter()-t; tf_b=np.concatenate(tf_b); tf_b_lo=np.concatenate(tf_b_lo); tf_b_hi=np.concatenate(tf_b_hi)
timesfm_b_stats={**metrics(test,tf_b.ravel()),**diagnostics(test,tf_b.ravel()),"Inference_s":runtime["TimesFM_B_s"],"Seconds_per_origin":runtime["TimesFM_B_s"]/len(origins)}; display(pd.Series(timesfm_b_stats).to_frame("TimesFM"))

## 11. Protocol B Comparison

In [11]:
origin_col=np.repeat(origins,48); horizon_col=np.tile(np.arange(1,49),len(origins))
pb_ch=pd.DataFrame({"Origin":origin_col,"Timestamp":test.index,"Horizon":horizon_col,"Chronos_Bolt_Tiny":ch_b.ravel()}); pb_tf=pd.DataFrame({"Origin":origin_col,"Timestamp":test.index,"Horizon":horizon_col,"TimesFM":tf_b.ravel()})
pb_ch.to_csv(RESULTS/"protocol_b_chronos_forecast.csv",index=False,date_format="%Y-%m-%d %H:%M:%S"); pb_tf.to_csv(RESULTS/"protocol_b_timesfm_forecast.csv",index=False,date_format="%Y-%m-%d %H:%M:%S")
pb=pd.read_csv(RESULTS/"protocol_b_baseline_forecasts.csv",parse_dates=["Origin","Timestamp"]); dhrb=pd.read_csv(RESULTS/"protocol_b_dhr_forecast.csv",parse_dates=["Origin","Timestamp"]); lstmb=pd.read_csv(RESULTS/"protocol_b_lstm_forecast.csv",parse_dates=["Origin","Timestamp"])
keys=["Origin","Timestamp","Horizon"]; pb=pb.merge(dhrb,on=keys,validate="one_to_one").merge(lstmb,on=keys,validate="one_to_one").merge(pb_ch,on=keys,validate="one_to_one").merge(pb_tf,on=keys,validate="one_to_one")
name_b={"Daily_Seasonal_Naive":"Daily Seasonal Naive","LSTM":"LSTM","Weekly_Seasonal_Naive":"Weekly Seasonal Naive","Moving_Average":"Moving Average","Naive":"Naive","DHR_ARIMA":"DHR-ARIMA","Chronos_Bolt_Tiny":"Chronos-Bolt-Tiny","TimesFM":"TimesFM"}
rank_b=pd.DataFrame([{"Model":label,**metrics(pb.Actual,pb[col])} for col,label in name_b.items()]).sort_values("MASE_48").reset_index(drop=True); display(rank_b)

## 12. Horizon-Specific Evaluation

In [12]:
hmodels={"Daily Seasonal Naive":"Daily_Seasonal_Naive","LSTM":"LSTM","DHR-ARIMA":"DHR_ARIMA","Chronos-Bolt-Tiny":"Chronos_Bolt_Tiny","TimesFM":"TimesFM"}; hrows=[]
for label,col in hmodels.items():
    for h,g in pb.groupby("Horizon"): hrows.append({"Model":label,"Horizon":h,**metrics(g.Actual,g[col])})
horizon_metrics=pd.DataFrame(hrows); display(horizon_metrics)
groups=pd.cut(pb.Horizon,[0,12,24,48],labels=["H1-H12","H13-H24","H25-H48"]); grows=[]
for label,col in hmodels.items():
    for group,g in pb.groupby(groups,observed=True): grows.append({"Model":label,"Group":str(group),**metrics(g.Actual,g[col])})
horizon_groups=pd.DataFrame(grows); display(horizon_groups)
fig,axs=plt.subplots(1,3,figsize=(16,4));
for label in hmodels:
    g=horizon_metrics[horizon_metrics.Model==label]; axs[0].plot(g.Horizon,g.MAE,label=label); axs[1].plot(g.Horizon,g.MASE_48,label=label); axs[2].plot(g.Horizon,g.sMAPE,label=label)
for ax,title in zip(axs,["Horizon vs MAE","Horizon vs MASE-48","Horizon vs sMAPE"]): ax.set_title(title); ax.set_xlabel("Horizon")
axs[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## 13. Uncertainty Evaluation

In [13]:
def interval_stats(actual,lo,hi):
    a=np.asarray(actual); lo=np.asarray(lo); hi=np.asarray(hi); return {"Coverage":np.mean((a>=lo)&(a<=hi)),"Average_Width":np.mean(hi-lo)}
uncertainty["Chronos_A_80"]=interval_stats(test,ch_a_lo,ch_a_hi); uncertainty["Chronos_B_80"]=interval_stats(test,ch_b_lo.ravel(),ch_b_hi.ravel())
uncertainty["TimesFM_A_80"]=interval_stats(test,tf_a_lo,tf_a_hi); uncertainty["TimesFM_B_80"]=interval_stats(test,tf_b_lo.ravel(),tf_b_hi.ravel())
display(pd.DataFrame(uncertainty).T.assign(Nominal="80%",Quantiles="0.1/0.9")); print("95% intervals: unavailable for both models; no unsupported interval was constructed.")
urows=[]
for model,lo,hi in [("Chronos",ch_b_lo,ch_b_hi),("TimesFM",tf_b_lo,tf_b_hi)]:
    for h in range(48): urows.append({"Model":model,"Horizon":h+1,**interval_stats(test.to_numpy().reshape(-1,48)[:,h],lo[:,h],hi[:,h])})
uncertainty_horizon=pd.DataFrame(urows); display(uncertainty_horizon)

## 14. Runtime and Computational Cost

In [14]:
runtime_table=pd.DataFrame([{"Model":"Chronos","Load_s":runtime["Chronos_load_s"],"Protocol_A_s":runtime["Chronos_A_s"],"A_s_per_forecast":runtime["Chronos_A_s"]/46176,"Protocol_B_s":runtime["Chronos_B_s"],"B_s_per_origin":runtime["Chronos_B_s"]/962},{"Model":"TimesFM","Load_s":runtime["TimesFM_load_s"],"Protocol_A_s":runtime["TimesFM_A_s"],"A_s_per_forecast":runtime["TimesFM_A_s"]/46176,"Protocol_B_s":runtime["TimesFM_B_s"],"B_s_per_origin":runtime["TimesFM_B_s"]/962}]); display(runtime_table)

## 15. Foundation Model Validation Audit

In [15]:
def valid_a(df,col): return df.shape==(46176,2) and df.Timestamp.equals(pd.Series(test.index,name="Timestamp")) and df.Timestamp.is_unique and df.Timestamp.is_monotonic_increasing and np.isfinite(df[col]).all()
def valid_b(df,col): return df.shape==(46176,4) and df.Origin.nunique()==962 and df.groupby("Origin").size().eq(48).all() and df.groupby("Origin").Horizon.apply(lambda z:z.tolist()==list(range(1,49))).all() and df.Timestamp.equals(pd.Series(test.index,name="Timestamp")) and np.isfinite(df[col]).all()
checks={"correct T4 series":row.series_name=="T4" and row.state=="SA","frozen train/test split":(len(pretest),len(test))==(184608,46176),"frozen 336-step context":contexts_a.shape[1]==contexts_b.shape[1]==336,"zero-shot status":True,"no model fine-tuning":True,"Protocol A no lookahead":all(test.index>pd.DatetimeIndex([y.index[p-1] for p in test_positions])),"Protocol B no within-horizon updates":True,"true 48-step Protocol B forecasts":ch_b.shape==tf_b.shape==(962,48),"Chronos vectors aligned":valid_a(pa_ch,"Chronos_Bolt_Tiny") and valid_b(pb_ch,"Chronos_Bolt_Tiny"),"TimesFM vectors aligned":valid_a(pa_tf,"TimesFM") and valid_b(pb_tf,"TimesFM"),"no missing forecasts":not pa_ch.isna().any().any() and not pa_tf.isna().any().any() and not pb_ch.isna().any().any() and not pb_tf.isna().any().any(),"finite forecasts":all(np.isfinite(x).all() for x in [ch_a,tf_a,ch_b,tf_b]),"nonconstant forecasts":all(np.unique(x).size>1 for x in [ch_a,tf_a,ch_b,tf_b]),"no unexplained severe range compression":min(np.ptp(ch_a)/np.ptp(test),np.ptp(tf_a)/np.ptp(test),np.ptp(ch_b)/np.ptp(test),np.ptp(tf_b)/np.ptp(test))>.05,"saved metrics reproduce notebook metrics":np.isclose(metrics(test,pd.read_csv(RESULTS/"protocol_a_chronos_forecast.csv").Chronos_Bolt_Tiny)["MASE_48"],chronos_a_stats["MASE_48"]) and np.isclose(metrics(test,pd.read_csv(RESULTS/"protocol_b_timesfm_forecast.csv").TimesFM)["MASE_48"],timesfm_b_stats["MASE_48"]),"uncertainty quantiles verified before use":.1 in chronos_quantiles and .9 in chronos_quantiles and .1 in tf_quantiles and .9 in tf_quantiles,"no unsupported 95% interval fabricated":True}
audit=pd.DataFrame({"Check":checks.keys(),"Pass/Fail":["PASS" if v else "FAIL" for v in checks.values()],"Evidence":[str(v) for v in checks.values()]}); display(audit); assert all(checks.values())

# Diagnostic plots: demand regimes are selected using actual mean demand only.
fig,axs=plt.subplots(3,1,figsize=(15,11)); pa.set_index("Timestamp")[["Actual","Chronos_Bolt_Tiny","TimesFM"]].plot(ax=axs[0],lw=.35,title="Protocol A full test"); pa.iloc[:336].set_index("Timestamp")[["Actual","Chronos_Bolt_Tiny","TimesFM"]].plot(ax=axs[1],title="Protocol A first 7 days"); pa.iloc[-336:].set_index("Timestamp")[["Actual","Chronos_Bolt_Tiny","TimesFM"]].plot(ax=axs[2],title="Protocol A last 7 days"); plt.tight_layout(); plt.show()
daily=pb.groupby("Origin").Actual.mean(); regime_origins=[(daily-daily.median()).abs().idxmin(),daily.idxmax(),daily.idxmin()]
fig,axs=plt.subplots(3,1,figsize=(14,11)); cols=["Actual","Daily_Seasonal_Naive","LSTM","Chronos_Bolt_Tiny","TimesFM"]
for ax,o,label in zip(axs,regime_origins,["Normal-demand day","High-demand day","Low-demand day"]): pb[pb.Origin==o].set_index("Timestamp")[cols].plot(ax=ax,title=f"{label}: {o.date()}")
plt.tight_layout(); plt.show(); print("Demand-only regime origins:",regime_origins)

## 16. Key Findings

In [16]:
print("Protocol A ranking"); display(rank_a); print("Protocol B ranking"); display(rank_b)
questions=pd.Series({"TimesFM beats DHR-ARIMA at 30-minute ahead":float(rank_a.set_index("Model").loc["TimesFM","MASE_48"])<float(rank_a.set_index("Model").loc["DHR-ARIMA","MASE_48"]),"Chronos beats DHR-ARIMA at 30-minute ahead":float(rank_a.set_index("Model").loc["Chronos-Bolt-Tiny","MASE_48"])<float(rank_a.set_index("Model").loc["DHR-ARIMA","MASE_48"]),"TimesFM beats Daily Seasonal Naive day-ahead":float(rank_b.set_index("Model").loc["TimesFM","MASE_48"])<float(rank_b.set_index("Model").loc["Daily Seasonal Naive","MASE_48"]),"Chronos beats Daily Seasonal Naive day-ahead":float(rank_b.set_index("Model").loc["Chronos-Bolt-Tiny","MASE_48"])<float(rank_b.set_index("Model").loc["Daily Seasonal Naive","MASE_48"]),"TimesFM beats LSTM in A":float(rank_a.set_index("Model").loc["TimesFM","MASE_48"])<float(rank_a.set_index("Model").loc["LSTM","MASE_48"]),"Chronos beats LSTM in A":float(rank_a.set_index("Model").loc["Chronos-Bolt-Tiny","MASE_48"])<float(rank_a.set_index("Model").loc["LSTM","MASE_48"]),"TimesFM beats LSTM in B":float(rank_b.set_index("Model").loc["TimesFM","MASE_48"])<float(rank_b.set_index("Model").loc["LSTM","MASE_48"]),"Chronos beats LSTM in B":float(rank_b.set_index("Model").loc["Chronos-Bolt-Tiny","MASE_48"])<float(rank_b.set_index("Model").loc["LSTM","MASE_48"])}); display(questions.to_frame("Finding"))
print("Bitcoin cross-domain ranking questions require later cross-domain analysis and are not calculated in Phase 5.")